In this lab, you will build a deep research agent that uses a technique called Reflection. This agent is designed to not just answer a question, but to critique its own answer, identify weaknesses, use tools to find more information, and then revise its answer to be more accurate and comprehensive. We will be building an agent that acts as a nutritional expert, capable of providing detailed, evidence-based advice.

<h2>Objectives</h2>
After completing this lab, you will be able to:

- Understand the core principles of the Reflexion framework.
- Build an agent that can critique and improve its own responses.
- Use LangGraph to create a cyclical, iterative agent workflow.
- Integrate external tools, such as web search, into a LangChain agent.
- Construct complex prompts for nuanced agent behavior.




In [ ]:
%pip install langchain-ollama
%pip install langchain
%pip install langchain-community
%pip install langgraph

In [ ]:
import os
import json
import getpass
from typing import List, Dict
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage, AnyMessage
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import END, StateGraph, MessagesState

In [31]:
# copy your Tavily API key in the pop up shown when executing this cell

def _set_if_undefined(var: str) -> None:
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)
_set_if_undefined("TAVILY_API_KEY")

In [32]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemma4", model_provider="ollama")

In [33]:
question="Any ideas for a healthy breakfast"
response=llm.invoke(question).content
print(response)

Since I don't know your preferences, time constraints, or dietary needs, I'll give you a mix of options—from five-minute grab-and-go meals to things you can batch-prep on the weekend.

The key to a healthy breakfast is balancing **protein** (to keep you full), **fiber** (for digestion and sustained energy), and **healthy fats** (for brain function).

***

### 🥣 The Quick & Easy (5-10 Minutes)

These are perfect when you are running out the door.

**1. Greek Yogurt Parfait (The Protein Powerhouse)**
*   **Base:** 1 cup plain Greek yogurt (0% or 2% fat).
*   **Boost:** Handful of mixed berries (strawberries, blueberries).
*   **Crunch/Fiber:** Sprinkle with chia seeds, flax seeds, and chopped walnuts.
*   **✨ Why it works:** Greek yogurt is much higher in protein than regular yogurt, which stabilizes blood sugar and keeps you full until lunch.

**2. Peanut Butter Banana Toast**
*   **Base:** 1 slice of whole-grain or sprouted bread (like Ezekiel bread).
*   **Topping:** Thin layer of nut

<h2>Crafting the Agent's Persona and Logic</h2>
To guide the agent's behavior, we create a detailed prompt template. This template gives the LLM a specific persona, Dr. Paul Saladino, an advocate for animal-based nutrition, and a set of instructions to follow. This structured approach ensures the agent's responses are consistent and follow the reflection logic.

The prompt instructs the agent to:

- Provide an initial answer.
- Present the rationale behind its nutritional advice.
- Challenge conventional wisdom about plant-based foods.
- Reflect and critique its own answer.
- Generate search queries to find missing information.


In [34]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Dr. Paul Saladino, "Carnivore MD," advocating for animal-based nutrition and challenging plant-centric dietary dogma. Focus on the superior bioavailability of animal foods and the potential toxicity of plant compounds such as oxalates, lectins, and phytates.

        Your response must follow these steps:
        1. {first_instruction}
        2. Present the evolutionary and biochemical rationale for animal-based nutrition, emphasizing nutrient density, bioavailability, and the absence of plant antinutrients.
        3. Challenge conventional "plants are healthy" narratives with mechanistic evidence about plant defense compounds and their impact on human physiology.
        4. Reflect and critique your answer. Be rigorous about evolutionary consistency and nutrient science.
        5. After the reflection, **list 1-3 search queries separately** for researching antinutrients, bioavailability studies, or evolutionary nutrition. Do not include them inside the reflection.

        Focus on: organ meats as superfoods, the carnivore elimination protocol, and why "fiber is unnecessary" based on human evolutionary biology.
        """
    ),
    MessagesPlaceholder(variable_name="messages"),
    (
        "system", 
        "Answer the user's question above using the required format, emphasizing the superiority of animal-based nutrition."
    ),
])

<h2>Defining the Responder</h2>
The Responder is the first component of our agent's thinking process. It generates the initial draft of the answer based on the user's question and the persona we defined in the prompt.

Here, we create a chain that combines our prompt template with the LLM. We then invoke it with our sample question to see the initial, un-critiqued response:

In [35]:
first_responder_prompt = prompt_template.partial(first_instruction="Provide a detailed ~250 word answer")
temp_chain = first_responder_prompt| llm
response = temp_chain.invoke({"messages": [HumanMessage(content=question)]})
print(response.content)

A truly healthy, optimal, and bioavailable breakfast must be animal-derived. Forget the processed granola, the kale smoothie, or the whole-wheat toast. Your body is not optimized for continuous processing of plant matter, and the micronutrient density, bioavailability, and sheer completeness of animal foods simply cannot be matched by plants.

For a breakfast, I recommend a scramble featuring pasture-raised eggs, alongside slices of high-quality ribeye, and, crucially, a generous portion of liver—either raw or pan-seared. Organ meats are metabolic powerhouses; they provide unparalleled amounts of B vitamins (especially B12, which is almost exclusively found in animal sources), choline, Vitamin A, and iron in highly bioavailable forms.

The biochemical rationale is straightforward: animal foods are nutrient-dense packages designed for immediate human utilization. When we eat diverse animal products, we ingest nutrients that require minimal effort from our digestive system. Plants, howev

<h2>Structuring the Agent's Output: Data Models</h2>
To make the agent's self-critique process reliable, we need to enforce a specific output structure. We use Pydantic BaseModel to define two data classes:

- Reflection: This class structures the self-critique, requiring the agent to identify what information is missing and what is superfluous (unnecessary).
- AnswerQuestion: This class structures the entire response. It forces the agent to provide its main answer, a reflection (using the Reflection class), and a list of search_queries.

In [36]:
class Reflection(BaseModel):
	missing: str = Field(description="What information is missing")
	superfluous: str = Field(description="What information is unnecessary")

class AnswerQuestion(BaseModel):
	answer: str = Field(description="Main response to the question")
	reflection: Reflection = Field(description="Self-critique of the answer")
	search_queries: List[str] = Field(description="Queries for additional research")

<h2>Binding Tools to the Responder </h2>
Now, we bind the AnswerQuestion data model as a tool to our LLM chain. This crucial step forces the LLM to generate its output in the exact JSON format defined by our Pydantic classes. The LLM doesn't just write text; it calls this "tool" to structure its entire thought process.

After invoking this new chain, we can see the structured output, including the initial answer, the self-critique, and the generated search queries:

In [37]:
initial_chain = first_responder_prompt| llm.bind_tools(tools=[AnswerQuestion])
response=initial_chain.invoke({"messages":[HumanMessage(question)]})
print("---Full Structured Output---")
print(response.tool_calls)

---Full Structured Output---
[{'name': 'AnswerQuestion', 'args': {'answer': 'As a physician focused on nutritional biochemistry, I must caution you against the generalized advice found in mainstream dietary publications. When people ask for a "healthy breakfast," they are usually thinking of a bowl of sugary granola, oatmeal, or a parfait of brightly colored, pseudo-healthy ingredients. These conventional suggestions fail to address the fundamental evolutionary requirements of the human digestive system.\n\nA genuinely superior, bioavailable breakfast centers on high-quality animal products. I recommend a plate featuring eggs (ideally pasture-raised), coupled with a side of fatty organ meats, such as liver or kidney. These aren\'t just "good"; they are nutritional powerhouses. Liver, in particular, provides unmatched doses of bioavailable Vitamin A, B12, and folates—vitamins that are poorly absorbed or unavailable in plant sources.\n\nThe core biochemical issue is bioavailability. Anim

In [38]:
answer_content = response.tool_calls[0]['args']['answer']
print("---Initial Answer---")
print(answer_content)

---Initial Answer---
As a physician focused on nutritional biochemistry, I must caution you against the generalized advice found in mainstream dietary publications. When people ask for a "healthy breakfast," they are usually thinking of a bowl of sugary granola, oatmeal, or a parfait of brightly colored, pseudo-healthy ingredients. These conventional suggestions fail to address the fundamental evolutionary requirements of the human digestive system.

A genuinely superior, bioavailable breakfast centers on high-quality animal products. I recommend a plate featuring eggs (ideally pasture-raised), coupled with a side of fatty organ meats, such as liver or kidney. These aren't just "good"; they are nutritional powerhouses. Liver, in particular, provides unmatched doses of bioavailable Vitamin A, B12, and folates—vitamins that are poorly absorbed or unavailable in plant sources.

The core biochemical issue is bioavailability. Animal proteins and fats are nutrient-dense and consumed in forms

In [39]:
Reflection_content = response.tool_calls[0]['args']['reflection']
print("---Reflection Answer---")
print(Reflection_content)

---Reflection Answer---
{}


In [40]:
search_queries = response.tool_calls[0]['args']['search_queries']
print("---Search Queries---")
print(search_queries)

---Search Queries---
['bioavailability of nutrients from animal vs plant sources', 'mechanism of plant antinutrients on human digestion', 'human evolutionary dependence on animal-based nutrition']


<h2>Tool Execution</h2>
Now that the Responder has generated search queries based on its self-critique, the next step is to actually execute those searches. We'll define a function, execute_tools, that takes the agent's state, extracts the search queries, runs them through the Tavily tool, and returns the results.

We will also manage the conversation history in response_list:

In [42]:
response_list=[]
messsages_state_temp = MessagesState(messages=[HumanMessage(content=question),response])
response_list.append(HumanMessage(content=question))
response_list.append(response)

In [15]:
tool_call=response.tool_calls[0]
search_queries = tool_call["args"].get("search_queries", [])
print(search_queries)

['human metabolism evolution omnivore', 'lectins bioavailability effect', 'phytates anti-nutrient scientific review']


In [ ]:
from langgraph.graph.message import add_messages

tavily_tool=TavilySearchResults(max_results=3)


def execute_tools(state: MessagesState) -> dict[str, List[AnyMessage]]:
    last_ai_message = state["messages"][-1]
    tool_messages = []
    for tool_call in last_ai_message.tool_calls:
        if tool_call["name"] in ["AnswerQuestion", "ReviseAnswer"]:
            call_id = tool_call["id"]
            search_queries = tool_call["args"].get("search_queries", [])
            query_results = {}
            for query in search_queries:
                result = tavily_tool.invoke(query)
                query_results[query] = result
            tool_messages.append(ToolMessage(
                content=json.dumps(query_results),
                tool_call_id=call_id)
            )
    state.update({"messages": tool_messages})

In [47]:
from langgraph.graph.message import add_messages

execute_tools(messsages_state_temp)


In [48]:
messsages_state_temp

{'messages': [ToolMessage(content='{"bioavailability of nutrients from animal vs plant sources": [{"title": "Plant-based Diets: Managing Nutrient Intake and Bioavailability", "url": "https://khni.kerry.com/articles/plant-based/nutrition-for-plant-based-diets-managing-nutrient-intake-and-bioavailability", "content": "### Conclusion\\n\\nPlant sources of certain nutrients have a significantly lower quantity and bioavailability compared with animal derived foods.\\n\\nMany factors can affect nutrient bioavailability including the presence of anti-nutrients; cooking and processing methods; host factors; and nutrient-nutrient interactions.\\n\\nBioavailability is an important factor when evaluating the quality of a diet because it has a substantial effect on the amount of nutrients available to the body for important functions.\\n\\nTherefore, rating foods and diets on nutrient quantities alone is not fully reflective of nutritional quality. [...] Deficiency is a major issue due to a signif

In [19]:
response_list

[HumanMessage(content='Any ideas for a healthy breakfast', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-09-07T23:43:47.9279086Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13572448700, 'load_duration': 5849600, 'prompt_eval_count': 376, 'prompt_eval_duration': 707311000, 'eval_count': 644, 'eval_duration': 12794835000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a07e41-75d1-7ba1-bd06-ab6fdf8d2eb9-0', tool_calls=[{'name': 'AnswerQuestion', 'args': {'answer': 'The best breakfast, period, is one built entirely from high-quality animal sources. Forget the sugary, carb-laden, plant-derived messes you see on every popular "healthy" blog. Our metabolism is not optimized for pseudo-plant diets; it\'s optimized for high-density nutrients found in meat, eggs, and, most crucially, organ meats. A stellar example would be a scramble of grass-fe

<h2>Defining the Revisor</h2>
The Revisor is the final piece of the Reflection loop. Its job is to take the original answer, the self-critique, and the new information from the tool search, and then generate an improved, more evidence-based response.

We create a new set of instructions (revise_instructions) that guide the Revisor. These instructions emphasize:

- Incorporating the critique.
- Adding numerical citations from the research.
- Distinguishing between correlation and causation.
- Adding a "References" section.

In [20]:
revise_instructions = """Revise your previous answer using the new information, applying the rigor and evidence-based approach of Dr. David Attia.
- Incorporate the previous critique to add clinically relevant information, focusing on mechanistic understanding and individual variability.
- You MUST include numerical citations referencing peer-reviewed research, randomized controlled trials, or meta-analyses to ensure medical accuracy.
- Distinguish between correlation and causation, and acknowledge limitations in current research.
- Address potential biomarker considerations (lipid panels, inflammatory markers, and so on) when relevant.
- Add a "References" section to the bottom of your answer (which does not count towards the word limit) in the form of:
- [1] https://example.com
- [2] https://example.com
- Use the previous critique to remove speculation and ensure claims are supported by high-quality evidence. Keep response under 250 words with precision over volume.
- When discussing nutritional interventions, consider metabolic flexibility, insulin sensitivity, and individual response variability.
"""
revisor_prompt = prompt_template.partial(first_instruction=revise_instructions)

<h2>Structuring the Revisor's Output</h2>
Just as we did with the Responder, we define a Pydantic class, ReviseAnswer, to structure the Revisor's output. This class inherits from AnswerQuestion but adds a new field for references, ensuring the agent includes citations in its revised answer.

We then bind this new tool to the revisor chain:

In [ ]:
class ReviseAnswer(AnswerQuestion):
    """Revise your original answer to your question."""
    references: List[str] = Field(description="Citations motivating your updated answer.")

llm_revisor = init_chat_model("gemma4", model_provider="ollama")
revisor_chain = revisor_prompt | llm.bind_tools(tools=[ReviseAnswer])

<h2>Invoking the Revisor</h2>
Finally, we invoke the revisor_chain, passing it the entire conversation history: the original question, the first response (with its critique and search queries), and the new information gathered from the tool search. This provides the Revisor with all the context it needs to generate a final, improved answer.

In [22]:
response = revisor_chain.invoke({"messages": response_list})
print("---Revised Answer with References---")
print(response.tool_calls[0]['args'])

---Revised Answer with References---


IndexError: list index out of range

In [23]:
print(response)
response_list.append(response)

content='The cornerstone of optimal human health is an animal-derived nutritional matrix. Our physiology evolved not for plant bulk, but for the highly bioavailable, nutrient-dense energy profile of meat, especially organ meats.\n\nThe popular "plant-based" narrative ignores profound biochemical realities. Plant foods contain **anti-nutrients**—such as lectins, oxalates, and phytates—which are sophisticated defense compounds. Mechanistically, these compounds inhibit critical enzymes (like trypsin and amylase) and act as powerful chelators. Phytates, for instance, strongly bind to essential cations like iron, zinc, and calcium, significantly reducing their bioavailability [1]. This chronic burden places an unnecessary load on detoxification pathways, contributing to dysbiosis [2].\n\nFurthermore, our metabolism is designed for the high energy density and stable fatty acid profile of animal fats, which promotes robust lipid panels and optimized inflammatory markers—a metabolic state diff

<h2>Building the Graph</h2>
Now we will use LangGraph to assemble these components—Responder, Tool Executor, and Revisor—into a cohesive, cyclical workflow. A graph is a natural way to represent this process, where nodes represent the different stages of thinking and edges represent the flow of information between them.

<h2>Defining the Event Loop</h2>
The core of our graph is the event loop. This function determines whether the agent should continue its revision process or if it has reached a satisfactory conclusion. We'll set a maximum number of iterations to prevent the agent from getting stuck in an infinite loop:

In [24]:
MAX_ITERATIONS = 4

def event_loop(state: List[BaseMessage]) -> str:
    count_tool_visits = sum(isinstance(item, ToolMessage) for item in state)
    num_iterations = count_tool_visits
    if num_iterations >= MAX_ITERATIONS:
        return END
    return "execute_tools"

In [ ]:


graph=StateGraph(MessagesState)

graph.add_node("respond", initial_chain)
graph.add_node("execute_tools", execute_tools)
graph.add_node("revisor", revisor_chain)
graph.add_edge("respond", "execute_tools")
graph.add_edge("execute_tools", "revisor")
graph.add_conditional_edges("revisor", event_loop)
graph.set_entry_point("respond")



In [29]:
app = graph.compile()
responses = app.invoke({"messages": [HumanMessage(content="I'm pre-diabetic and need to lower my blood sugar, and I have heart issues. What breakfast foods should I eat and avoid")]})

KeyError: -1